# 📚 Book Sales Data Analysis
*Enhanced visualizations — clear, multi-dimensional insights*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

# ── Global theme ──────────────────────────────────────────────────────────────
BG = '#F7F9FC'
GENRE_PAL  = {'genre fiction': '#2D6A9F', 'fiction': '#4EA8DE',
               'nonfiction': '#56CFE1', 'children': '#80FFDB'}
AUTHOR_PAL = {'Novice': '#4EA8DE', 'Intermediate': '#2D6A9F',
              'Excellent': '#F4A261', 'Famous': '#E76F51'}
RATING_ORDER = ['Novice', 'Intermediate', 'Excellent', 'Famous']

sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({
    'axes.facecolor': BG, 'figure.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.labelsize': 11, 'xtick.labelsize': 9, 'ytick.labelsize': 9
})


## 1. Load & Clean Data

In [ ]:
df = pd.read_csv('Books_Data_Clean.csv')
df = df[df['Publishing Year'] > 1900]
df.dropna(subset='Book Name', inplace=True)
df['Publisher '] = df['Publisher '].str.strip()

print(f"Dataset shape: {df.shape}")
df.head()


In [ ]:
df.describe()


In [ ]:
print("Missing values:")
df.isna().sum()[df.isna().sum() > 0]


## 2. Publishing Trends & Genre Landscape

In [ ]:
fig1, axes = plt.subplots(2, 3, figsize=(18, 10))
fig1.suptitle('Book Sales — Publishing Trends & Genre Landscape', fontsize=16, fontweight='bold', y=0.98)
gs  = gridspec.GridSpec(2, 3, figure=fig1, hspace=0.45, wspace=0.38)
fig1.clear()

# ── 1a: Stacked area — units sold by genre over years ──────────────────────
ax1a = fig1.add_subplot(gs[0, :2])
yr_genre = (df.groupby(['Publishing Year', 'genre'])['units sold']
              .sum().unstack(fill_value=0))
yr_genre = yr_genre[yr_genre.index >= 1970]
genres = yr_genre.columns.tolist()
ax1a.stackplot(yr_genre.index, yr_genre.T.values,
               labels=genres,
               colors=[GENRE_PAL.get(g, '#AAB4C8') for g in genres], alpha=0.85)
ax1a.set_title('Total Units Sold by Genre Over Time')
ax1a.set_xlabel('Publishing Year'); ax1a.set_ylabel('Units Sold')
ax1a.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1a.legend(loc='upper left', fontsize=8, framealpha=0.7)

# ── 1b: Donut — genre distribution ─────────────────────────────────────────
ax1b = fig1.add_subplot(gs[0, 2])
g_counts = df['genre'].value_counts()
wedges, texts, autotexts = ax1b.pie(
    g_counts, labels=g_counts.index,
    colors=[GENRE_PAL.get(g, '#AAB4C8') for g in g_counts.index],
    autopct='%1.1f%%', startangle=140, pctdistance=0.78,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2))
for at in autotexts: at.set_fontsize(8)
ax1b.set_title('Genre Distribution')

# ── 1c: KDE — publishing year density by genre ─────────────────────────────
ax1c = fig1.add_subplot(gs[1, :2])
for genre, grp in df.groupby('genre'):
    sns.kdeplot(grp['Publishing Year'], ax=ax1c, label=genre,
                color=GENRE_PAL.get(genre, '#AAB4C8'), linewidth=2, fill=True, alpha=0.25)
ax1c.set_title('Publishing Year Density by Genre')
ax1c.set_xlabel('Publishing Year'); ax1c.set_ylabel('Density')
ax1c.legend(fontsize=8)

# ── 1d: Horizontal bar — avg units sold per genre ──────────────────────────
ax1d = fig1.add_subplot(gs[1, 2])
avg_units = df.groupby('genre')['units sold'].mean().sort_values()
bars = ax1d.barh(avg_units.index, avg_units.values,
                 color=[GENRE_PAL.get(g, '#AAB4C8') for g in avg_units.index],
                 edgecolor='white')
ax1d.set_title('Avg Units Sold by Genre')
ax1d.set_xlabel('Avg Units Sold')
ax1d.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars, avg_units.values):
    ax1d.text(val + 30, bar.get_y() + bar.get_height()/2,
              f'{val:,.0f}', va='center', fontsize=8)

plt.savefig('fig1_trends_genre.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Pricing, Ratings & Revenue

In [ ]:
fig2 = plt.figure(figsize=(18, 11))
fig2.suptitle('Book Sales — Pricing, Ratings & Revenue Analysis', fontsize=16, fontweight='bold', y=0.98)
gs2 = gridspec.GridSpec(2, 3, figure=fig2, hspace=0.45, wspace=0.38)

# ── 2a: Scatter — price vs units sold (genre-coloured) ─────────────────────
ax2a = fig2.add_subplot(gs2[0, :2])
for genre, grp in df.groupby('genre'):
    ax2a.scatter(grp['sale price'], grp['units sold'], label=genre, alpha=0.6, s=40,
                 color=GENRE_PAL.get(genre, '#AAB4C8'), edgecolors='white', linewidth=0.5)
mask = df['sale price'].notna() & df['units sold'].notna()
z = np.polyfit(df.loc[mask, 'sale price'], df.loc[mask, 'units sold'], 1)
xline = np.linspace(df['sale price'].min(), df['sale price'].max(), 200)
ax2a.plot(xline, np.poly1d(z)(xline), '--', color='#E76F51', linewidth=1.8, label='Trend')
ax2a.set_title('Sale Price vs Units Sold (by Genre)')
ax2a.set_xlabel('Sale Price ($)'); ax2a.set_ylabel('Units Sold')
ax2a.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax2a.legend(fontsize=8)

# ── 2b: Violin — book avg rating by author rating ──────────────────────────
ax2b = fig2.add_subplot(gs2[0, 2])
order_present = [r for r in RATING_ORDER if r in df['Author_Rating'].unique()]
sns.violinplot(data=df, x='Author_Rating', y='Book_average_rating',
               order=order_present, hue='Author_Rating',
               palette=AUTHOR_PAL, ax=ax2b, inner='quartile', cut=0, legend=False)
ax2b.set_title('Book Rating Distribution\nby Author Rating')
ax2b.set_xlabel('Author Rating'); ax2b.set_ylabel('Avg Book Rating')

# ── 2c: Scatter — avg rating vs ratings count (colour = units sold) ────────
ax2c = fig2.add_subplot(gs2[1, :2])
sc = ax2c.scatter(df['Book_average_rating'], df['Book_ratings_count'],
                  c=df['units sold'], cmap='YlOrRd', alpha=0.7, s=35, edgecolors='none')
cbar = fig2.colorbar(sc, ax=ax2c)
cbar.set_label('Units Sold', fontsize=9)
cbar.ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax2c.set_title('Book Avg Rating vs Ratings Count\n(color = Units Sold)')
ax2c.set_xlabel('Average Rating'); ax2c.set_ylabel('Ratings Count')
ax2c.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# ── 2d: Grouped bar — avg gross sales & publisher revenue by genre ──────────
ax2d = fig2.add_subplot(gs2[1, 2])
rev_genre = df.groupby('genre')[['gross sales', 'publisher revenue']].mean()
x = np.arange(len(rev_genre)); w = 0.35
ax2d.bar(x - w/2, rev_genre['gross sales'],     w, label='Gross Sales',    color='#2D6A9F', alpha=0.85)
ax2d.bar(x + w/2, rev_genre['publisher revenue'],w, label='Publisher Rev', color='#F4A261', alpha=0.85)
ax2d.set_xticks(x); ax2d.set_xticklabels(rev_genre.index, rotation=15, ha='right')
ax2d.set_title('Avg Revenue by Genre'); ax2d.set_ylabel('Amount ($)')
ax2d.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x):,}'))
ax2d.legend(fontsize=8)

plt.savefig('fig2_pricing_ratings.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Authors, Publishers & Correlations

In [ ]:
fig3 = plt.figure(figsize=(18, 11))
fig3.suptitle('Book Sales — Authors, Publishers & Key Correlations', fontsize=16, fontweight='bold', y=0.98)
gs3 = gridspec.GridSpec(2, 2, figure=fig3, hspace=0.45, wspace=0.38)

# ── 3a: Horizontal bar — top 15 authors by gross sales ─────────────────────
ax3a = fig3.add_subplot(gs3[0, 0])
top_authors = df.groupby('Author')['gross sales'].sum().sort_values(ascending=False).head(15)
colors_grad = plt.cm.Blues_r(np.linspace(0.2, 0.7, len(top_authors)))
bars = ax3a.barh(range(len(top_authors)), top_authors.values[::-1], color=colors_grad, edgecolor='white')
ax3a.set_yticks(range(len(top_authors)))
labels = [a[:25]+'…' if len(a) > 25 else a for a in top_authors.index[::-1]]
ax3a.set_yticklabels(labels, fontsize=7.5)
ax3a.set_title('Top 15 Authors by\nTotal Gross Sales')
ax3a.set_xlabel('Total Gross Sales ($)')
ax3a.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x):,}'))
for bar, val in zip(bars, top_authors.values[::-1]):
    ax3a.text(val + 100, bar.get_y() + bar.get_height()/2, f'${val:,.0f}', va='center', fontsize=6.5)

# ── 3b: Stacked bar — genre mix by top publisher ───────────────────────────
ax3b = fig3.add_subplot(gs3[0, 1])
top_pubs = df['Publisher '].value_counts().head(6).index
pub_genre = (df[df['Publisher '].isin(top_pubs)]
               .groupby(['Publisher ', 'genre']).size().unstack(fill_value=0))
pub_genre_pct = pub_genre.div(pub_genre.sum(axis=1), axis=0) * 100
pub_genre_pct.plot(kind='barh', stacked=True, ax=ax3b,
                   color=[GENRE_PAL.get(g, '#AAB4C8') for g in pub_genre_pct.columns],
                   edgecolor='white', width=0.7)
ax3b.set_title('Genre Mix by Top Publisher\n(% of titles)')
ax3b.set_xlabel('% of Titles'); ax3b.set_ylabel('')
ax3b.set_yticklabels([p[:28]+'…' if len(p) > 28 else p for p in pub_genre_pct.index], fontsize=7.5)
ax3b.legend(fontsize=7.5, loc='lower right', framealpha=0.7)
ax3b.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

# ── 3c: Boxplot — units sold by author rating ──────────────────────────────
ax3c = fig3.add_subplot(gs3[1, 0])
order_present = [r for r in RATING_ORDER if r in df['Author_Rating'].unique()]
box_data = [df[df['Author_Rating'] == r]['units sold'].values for r in order_present]
bp = ax3c.boxplot(box_data, tick_labels=order_present, patch_artist=True,
                  medianprops=dict(color='black', linewidth=2),
                  flierprops=dict(marker='o', markersize=3, alpha=0.4))
for patch, r in zip(bp['boxes'], order_present):
    patch.set_facecolor(AUTHOR_PAL.get(r, '#AAB4C8')); patch.set_alpha(0.85)
ax3c.set_title('Units Sold Distribution\nby Author Rating')
ax3c.set_xlabel('Author Rating'); ax3c.set_ylabel('Units Sold')
ax3c.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# ── 3d: Correlation heatmap ────────────────────────────────────────────────
ax3d = fig3.add_subplot(gs3[1, 1])
num_cols = ['Publishing Year', 'Book_average_rating', 'Book_ratings_count',
            'gross sales', 'publisher revenue', 'sale price', 'units sold']
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap=sns.diverging_palette(220, 20, as_cmap=True),
            center=0, annot=True, fmt='.2f', ax=ax3d,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 8})
ax3d.set_title('Correlation Heatmap\n(Numeric Variables)')
ax3d.set_xticklabels(ax3d.get_xticklabels(), rotation=30, ha='right', fontsize=8)
ax3d.set_yticklabels(ax3d.get_yticklabels(), rotation=0, fontsize=8)

plt.savefig('fig3_authors_publishers.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Summary Statistics

In [ ]:
print("=== Top 10 Authors by Gross Sales ===")
print(df.groupby('Author')['gross sales'].sum().sort_values(ascending=False).head(10).to_string())
print()
print("=== Avg Metrics by Genre ===")
print(df.groupby('genre')[['units sold','gross sales','sale price','Book_average_rating']].mean().round(2).to_string())
print()
print("=== Avg Rating Count by Author Rating ===")
print(df.groupby('Author_Rating')['Book_ratings_count'].mean().sort_values(ascending=False).round(0).to_string())
